In [0]:
# Gold answers questions such as:
    # 1. What is the average power?
    # 2. What is the average flow?
    # 3. What is the average temperature difference?
    # 4. How many valid records do we have?
    # 5. What is the daily system performance?

# Build 2 Gold tables:

# Table A: Interval KPI
# One row per timestamp snapshot
# Name: hvacapp_dev3.curated.gold_hvac_interval_kpi

# Table B: Daily KPI
# One row per day
# Name: hvacapp_dev3.curated.gold_hvac_daily_kpi

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hvacapp_dev3.curated;

In [0]:
%sql

-- This table is built from silver_telemetry_clean.
-- Each row represents one clean HVAC system snapshot.
-- Example:
-- event_ts	| supply_temp_c	| return_temp_c	| delta_temp_c | flow_lpm |	power_kw
-- 2026-03-27 10:01:00 |	8.5	| 12.0 |	3.5 |	780	| 350
-- This is already dashboard-ready.

CREATE OR REPLACE TABLE hvacapp_dev3.curated.gold_hvac_interval_kpi (
  event_ts TIMESTAMP,
  site_id STRING,
  building_id STRING,
  hvac_system_id STRING,
  supply_temp_c DOUBLE,
  return_temp_c DOUBLE,
  delta_temp_c DOUBLE,
  flow_lpm DOUBLE,
  power_kw DOUBLE,
  pump_speed_pct DOUBLE,
  record_status STRING,
  created_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
-- Insert data from silver_clean into Gold Interval KPI

DELETE FROM hvacapp_dev3.curated.gold_hvac_interval_kpi;

INSERT INTO hvacapp_dev3.curated.gold_hvac_interval_kpi
SELECT
  event_ts,
  site_id,
  building_id,
  equipment_id AS hvac_system_id,
  chw_supply_temp_c AS supply_temp_c,
  chw_return_temp_c AS return_temp_c,
  (chw_return_temp_c - chw_supply_temp_c) AS delta_temp_c,
  chw_flow_lpm AS flow_lpm,
  hvac_power_kw AS power_kw,
  pump_speed_pct,
  'VALID' AS record_status,
  current_timestamp() AS created_ts
FROM hvacapp_dev3.refined.silver_telemetry_clean;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.curated.gold_hvac_interval_kpi
ORDER BY event_ts DESC;

In [0]:
%sql
-- Now create daily KPI table, data is from gold_interval

CREATE OR REPLACE TABLE hvacapp_dev3.curated.gold_hvac_daily_kpi (
  event_date DATE,
  site_id STRING,
  building_id STRING,
  hvac_system_id STRING,
  avg_supply_temp_c DOUBLE,
  avg_return_temp_c DOUBLE,
  avg_delta_temp_c DOUBLE,
  avg_flow_lpm DOUBLE,
  avg_power_kw DOUBLE,
  avg_pump_speed_pct DOUBLE,
  record_count BIGINT,
  created_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
-- now insert gold interval kpi data into daily kpi table

DELETE FROM hvacapp_dev3.curated.gold_hvac_daily_kpi;

INSERT INTO hvacapp_dev3.curated.gold_hvac_daily_kpi
SELECT
  CAST(event_ts AS DATE) AS event_date,
  site_id,
  building_id,
  hvac_system_id,
  AVG(supply_temp_c) AS avg_supply_temp_c,
  AVG(return_temp_c) AS avg_return_temp_c,
  AVG(delta_temp_c) AS avg_delta_temp_c,
  AVG(flow_lpm) AS avg_flow_lpm,
  AVG(power_kw) AS avg_power_kw,
  AVG(pump_speed_pct) AS avg_pump_speed_pct,
  COUNT(*) AS record_count,
  current_timestamp() AS created_ts
FROM hvacapp_dev3.curated.gold_hvac_interval_kpi
GROUP BY
  CAST(event_ts AS DATE),
  site_id,
  building_id,
  hvac_system_id;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.curated.gold_hvac_daily_kpi
ORDER BY event_date DESC;

In [0]:
# gold_hvac_interval_kpi
        # detailed charts over time
        # trend lines
        # time-series analysis

# gold_hvac_daily_kpi
        # daily summary
        # management reporting
        # dashboard cards
        # average daily performance